In [ ]:
from fastapi import APIRouter, Depends, HTTPException
from sqlalchemy.orm import Session
from core.database import get_db
from schemas.post import PostCreate, PostUpdate, PostResponse
from services import posts_service
from dependencies.auth import get_current_user

router = APIRouter(prefix="/posts", tags=["Posts"])


@router.get("/")
def get_posts(page: int = 1, size: int = 10, db: Session = Depends(get_db)):
    return posts_service.get_all_posts(db, page, size)


@router.get("/{post_id}", response_model=PostResponse)
def get_post(post_id: int, db: Session = Depends(get_db)):
    post = posts_service.get_post_by_id(db, post_id)

    if not post:
        raise HTTPException(status_code=404, detail="Post not found")

    return post


@router.post("/", response_model=PostResponse)
def create_post(
    post: PostCreate,
    db: Session = Depends(get_db),
    user=Depends(get_current_user)
):
    return posts_service.create_post(db, post, user)


@router.put("/{post_id}", response_model=PostResponse)
def update_post(
    post_id: int,
    post: PostUpdate,
    db: Session = Depends(get_db),
    user=Depends(get_current_user)
):
    updated = posts_service.update_post(db, post_id, post, user)

    if not updated:
        raise HTTPException(status_code=404, detail="Post not found")

    return updated


@router.delete("/{post_id}")
def delete_post(
    post_id: int,
    db: Session = Depends(get_db),
    user=Depends(get_current_user)
):
    result = posts_service.delete_post(db, post_id, user)

    if not result:
        raise HTTPException(status_code=404, detail="Post not found")

    return {"message": "Deleted successfully"}